# AIMENG D0：Google Colab 训练入口

目标：在 Google Colab GPU 上对 AIMENG 的 **8,192 神经元、32 维、Top-K 128** 配置做真实语料基线训练。

**断线保护：** 模型、优化器状态和已完成步数每 25 步原子写入 Google Drive 的 `training_state.pt`。运行时重置后，重新运行 Notebook 会检测检查点并续训。请始终把输出目录放在已挂载的 Drive 中，不要清理该目录。训练完成后，`diffusion_checkpoint.pt` 用于生成测试，`report.json` 保存指标，`training.log` 保存完整控制台日志。

**仓库资料：** 训练代码、Notebook、参数与数据说明都在 GitHub 仓库。数据和模型权重放 Drive，不把大文件塞进 Git 历史；小型报告/日志可以下载后再提交仓库。Colab 免费 GPU 配额和可用时长动态变化，不保证每天固定刷新或固定五小时。

首次运行会通过仓库脚本准备最多 50,000 条中文指令/回答样本，保存到 Drive。它是中文格式与训练流程的起点，**不是充足的语言教材，也不是思考能力证明**。完整语料课程规划见 `docs/LANGUAGE_CURRICULUM.md`。若使用自有语料，请确保拥有合法使用权，并将 UTF-8 文件放在 `MyDrive/aimeng_corpus.txt`。

In [ ]:
import os, subprocess, sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('No GPU runtime detected. In Colab choose Runtime > Change runtime type > GPU before training; this notebook is intended for GPU training.')

In [ ]:
# Pull the latest committed version of this branch on every notebook restart.
REPO = '/content/aimeng'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--branch', 'feat/google-colab-d-training', '--single-branch', 'https://github.com/bslsjdk/aimeng.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', 'feat/google-colab-d-training'], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', 'feat/google-colab-d-training'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/feat/google-colab-d-training'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
print('Repository ready:', REPO)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RAW_CORPUS_PATH = '/content/drive/MyDrive/aimeng_corpus_raw.txt'
SPLIT_DIR = '/content/drive/MyDrive/aimeng_language_splits'
CORPUS_PATH = os.path.join(SPLIT_DIR, 'train.txt')
VAL_PATH = os.path.join(SPLIT_DIR, 'validation.txt')
TEST_PATH = os.path.join(SPLIT_DIR, 'test.txt')
RUN_ROOT = '/content/drive/MyDrive/aimeng_runs'
os.makedirs(RUN_ROOT, exist_ok=True)

if not os.path.isfile(RAW_CORPUS_PATH):
    print('Preparing a bounded Chinese instruction baseline (up to 50,000 examples).')
    print('This is a starter set for pipeline/format checks, NOT a sufficient full language curriculum.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
    subprocess.run([
        sys.executable, f'{REPO}/scripts/prepare_speaking_corpus.py',
        '--dataset', 'BelleGroup/train_0.5M_CN',
        '--split', 'train',
        '--output', RAW_CORPUS_PATH,
        '--max-records', '50000',
        '--seed', '7'
    ], cwd=REPO, check=True)

with open(RAW_CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus_chars = sum(len(line) for line in f)
if corpus_chars < 10000:
    raise ValueError('Corpus appears too small. Check the data source or supply a larger UTF-8 corpus.')
print('Raw corpus ready:', RAW_CORPUS_PATH, '| characters:', corpus_chars)
if not (os.path.isfile(CORPUS_PATH) and os.path.isfile(VAL_PATH) and os.path.isfile(TEST_PATH)):
    subprocess.run([sys.executable, f'{REPO}/scripts/prepare_language_splits.py', '--input', RAW_CORPUS_PATH, '--output-dir', SPLIT_DIR, '--seed', '7'], cwd=REPO, check=True)
print('Record-level split files:', CORPUS_PATH, VAL_PATH, TEST_PATH)
print('Keep test.txt sealed until model/parameters are finalized; do not train on it.')
print('For the full language curriculum, see docs/LANGUAGE_CURRICULUM.md. Do not interpret this starter set as sufficient.')


## D0 预检
先用真实语料跑 20 步，验证 GPU、数据读取、前向/反向传播和检查点保存路径。这个步骤只验证训练链路，不代表模型已掌握语言。预检成功后再运行正式训练。

In [ ]:
SMOKE_OUT = os.path.join(RUN_ROOT, 'D0_preflight_record_splits')
cmd = [sys.executable, f'{REPO}/scripts/train_diffusion_text.py',
       '--text', CORPUS_PATH, '--validation-text', VAL_PATH, '--neurons', '8192', '--width', '32',
       '--active-k', '128', '--fanout', '8', '--max-steps', '4',
       '--context', '64', '--batch-size', '1', '--steps', '20',
       '--checkpoint-every', '5', '--log-every', '1', '--memory-stop-mib', '8192',
       '--output', SMOKE_OUT]
if os.path.isfile(os.path.join(SMOKE_OUT, 'training_state.pt')):
    cmd.append('--resume')
print('Running:', ' '.join(cmd))
log_path = os.path.join(SMOKE_OUT, 'training.log')
with open(log_path, 'a', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = proc.wait()
if code:
    raise subprocess.CalledProcessError(code, cmd)
print('Persistent preflight log:', log_path)


## D0 正式训练

默认 500 步。训练状态每 25 步原子保存到 Drive；如果 Colab 断线，重新连接 Drive 后重跑此单元格，会从最后一次完整检查点继续。若要延长训练，把 `TRAIN_STEPS` 调大即可，保持相同语料、输出目录和核心参数。更换语料或核心参数时，请使用新的输出目录，避免错误续训。

In [ ]:
TRAIN_STEPS = 500
D0_OUT = os.path.join(RUN_ROOT, 'D0_8192n_w32_k128_record_splits')
os.makedirs(D0_OUT, exist_ok=True)
cmd = [sys.executable, f'{REPO}/scripts/train_diffusion_text.py',
       '--text', CORPUS_PATH, '--validation-text', VAL_PATH, '--neurons', '8192', '--width', '32',
       '--active-k', '128', '--fanout', '8', '--max-steps', '4',
       '--context', '64', '--batch-size', '1', '--steps', str(TRAIN_STEPS),
       '--checkpoint-every', '25', '--log-every', '25', '--memory-stop-mib', '8192',
       '--output', D0_OUT]
if os.path.isfile(os.path.join(D0_OUT, 'training_state.pt')):
    cmd.append('--resume')
print('Training config: neurons=8192, width=32, active_k=128, fanout=8, propagation_steps=4, batch_size=1')
log_path = os.path.join(D0_OUT, 'training.log')
with open(log_path, 'a', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = proc.wait()
if code:
    raise subprocess.CalledProcessError(code, cmd)
print('Checkpoint:', os.path.join(D0_OUT, 'diffusion_checkpoint.pt'))
print('Resume state:', os.path.join(D0_OUT, 'training_state.pt'))
print('Report:', os.path.join(D0_OUT, 'report.json'))
print('Persistent log:', log_path)


In [ ]:
import json, os
report_path = os.path.join(D0_OUT, 'report.json')
with open(report_path, encoding='utf-8') as f:
    report = json.load(f)
print(json.dumps(report, ensure_ascii=False, indent=2))
print('\nInterpretation: lower held-out validation loss is an initial signal, not proof of broad language competence. Keep report.json and training.log with the checkpoint.')
print('All durable artifacts are in Google Drive:', D0_OUT)
